<a href="https://colab.research.google.com/github/Naveed-Bhutto/Assignment_PIAIC/blob/main/Project_1_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -Uq langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 25.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
import sqlite3  # Import the sqlite3 library

# Initialize Database (SQLite in this example)
db_path = "conversation_history.db"  # Choose your database path
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS conversations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_input TEXT,
        assistant_response TEXT
    )
''')
conn.commit()

# Initialize LLM
try:
    api_key = userdata.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("GEMINI_API_KEY not found. Please set the environment variable or in userdata.")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-exp",  # You can use gemini-pro for potentially better performance
        temperature=0.4,  # Adjust temperature for creativity vs. accuracy
        max_output_tokens=256,  # Adjust max tokens for desired response length
        api_key=api_key
    )

    # Initialize memory and prompt template
    memory = ConversationBufferMemory()
    prompt_template = PromptTemplate(
        input_variables=["history", "input"],
        template="""
        You are a helpful chat assistant.  You can access past conversation history from the database.
        {history}
        User: {input}
        Assistant:
        """
    )

    # Initialize conversation chain
    conversation = ConversationChain(
        llm=llm,
        memory=memory,
        prompt=prompt_template,
    )

    # Main interaction loop
    while True:
        user_input = input("User: ")
        if user_input.lower() == "exit":
            break
        try:
            response = conversation.predict(input=user_input)
            print(f"Assistant: {response}")

            # Store conversation in the database
            cursor.execute("INSERT INTO conversations (user_input, assistant_response) VALUES (?, ?)", (user_input, response))
            conn.commit()

        except Exception as e:
            print(f"An error occurred: {e}")

    # Retrieve Full History from the database
    cursor.execute("SELECT user_input, assistant_response FROM conversations")
    history_from_db = cursor.fetchall()
    print("Full conversation history (from database):")
    for user_input, assistant_response in history_from_db:
      print(f"User: {user_input}")
      print(f"Assistant: {assistant_response}")

except ValueError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the database connection in the finally block
    conn.close()